# Geospatial Analysis and Risk Mapping
This notebook performs geospatial analysis and feature extraction to understand localized pollution patterns.

In [ ]:
import sys
import os
# Set working directory to project root
os.chdir('..')
sys.path.append(os.path.abspath('.'))

### Step 1: Run Geospatial Analysis, Feature Extraction, and GeoJSON Generation

In [ ]:
from src.geospatial_analysis import main as run_geospatial
from src.feature_extraction import main as run_features
from src.generate_geojson import main as run_geojson

print("--- Running Geospatial Analysis ---")
run_geospatial()

print("\n--- Running Feature Extraction ---")
run_features()

print("\n--- Running GeoJSON Generation ---")
run_geojson()


### Step 2: Visualization

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import json

os.makedirs('notebooks/outputs', exist_ok=True)

# Load data for visualization
gdf = gpd.read_file('data/final/karnataka_risk_zones.geojson')
with open('data/final/geospatial_analysis_results.json', 'r') as f:
    results = json.load(f)


#### Moran's I and LISA Cluster Map

In [ ]:
print(f"Moran's I Statistic: {results['morans_i']['statistic']:.4f}")
print(f"p-value: {results['morans_i']['p_value']:.4f}")
print(f"Interpretation: {results['morans_i']['interpretation']}")

fig, ax = plt.subplots(figsize=(10, 8))
color_map = {
    'High-High hotspot': 'red',
    'Low-Low coldspot': 'blue',
    'High-Low outlier': 'lightblue',
    'Low-High outlier': 'pink',
    'Not Significant': 'lightgrey'
}
gdf['color'] = gdf['lisa_cluster'].map(color_map)
gdf.plot(color=gdf['color'], edgecolor='k', ax=ax)

# Add legend manually
import matplotlib.patches as mpatches
handles = [mpatches.Patch(color=v, label=k) for k, v in color_map.items()]
ax.legend(handles=handles, title='LISA Clusters', loc='lower right')
ax.set_title("LISA Cluster Map for PM2.5 in Karnataka")
ax.axis('off')

plt.savefig('notebooks/outputs/lisa_cluster_map.png', bbox_inches='tight')
plt.show()

#### Choropleth Map of Mean PM2.5

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
gdf.plot(column='mean_pm25', cmap='OrRd', edgecolor='k', legend=True, ax=ax,
         legend_kwds={'label': "Mean PM2.5 (µg/m³)"})
ax.set_title("Mean PM2.5 Concentration by District")
ax.axis('off')

plt.savefig('notebooks/outputs/pm25_choropleth.png', bbox_inches='tight')
plt.show()

#### Top 10 Most Polluted Districts

In [ ]:
top10 = gdf.sort_values(by='mean_pm25', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(top10['district'], top10['mean_pm25'], color='coral')
ax.set_title('Top 10 Most Polluted Districts (Mean PM2.5)')
ax.set_ylabel('Mean PM2.5 (µg/m³)')
ax.set_xlabel('District')
plt.xticks(rotation=45)

plt.savefig('notebooks/outputs/top10_polluted_districts.png', bbox_inches='tight')
plt.show()

#### Urban vs Rural and Seasonal Pattern Results

In [ ]:
print("--- Urban vs Rural Comparison ---")
if 'urban_vs_rural' in results:
    print(f"Urban Mean PM2.5: {results['urban_vs_rural']['urban_mean_pm25']:.2f}")
    print(f"Rural Mean PM2.5: {results['urban_vs_rural']['rural_mean_pm25']:.2f}")
    print(f"Mann-Whitney U p-value: {results['urban_vs_rural']['p_value']:.4f}")
else:
    print("Urban/Rural data not available.")

print("\n--- Seasonal Pattern Analysis ---")
if 'peak_month' in results:
    print(f"Peak pollution month: {results['peak_month']}")
    month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun', 7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
    print(f"({month_names.get(results['peak_month'], 'Unknown')})")
else:
    print("Seasonal data not available.")
